In [ ]:
# cell 1 — setup
!pip install -q torchvision
import json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}, device={DEVICE}")

In [ ]:
# cell 2 — data
data_root = Path("./data")
scratch_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
scratch_eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
resnet_train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
resnet_eval_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

scratch_full = datasets.CIFAR10(data_root, train=True, download=True, transform=scratch_train_tf)
scratch_val_source = datasets.CIFAR10(data_root, train=True, download=False, transform=scratch_eval_tf)
resnet_full = datasets.CIFAR10(data_root, train=True, download=False, transform=resnet_train_tf)
resnet_val_source = datasets.CIFAR10(data_root, train=True, download=False, transform=resnet_eval_tf)
generator = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(50_000), [45_000, 5_000], generator=generator)

def subset(dataset, indices):
    return torch.utils.data.Subset(dataset, list(indices))

scratch_train_loader = DataLoader(subset(scratch_full, train_idx), batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
scratch_val_loader = DataLoader(subset(scratch_val_source, val_idx), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
scratch_test_loader = DataLoader(datasets.CIFAR10(data_root, train=False, download=True, transform=scratch_eval_tf), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

resnet_train_loader = DataLoader(subset(resnet_full, train_idx), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
resnet_val_loader = DataLoader(subset(resnet_val_source, val_idx), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
resnet_test_loader = DataLoader(datasets.CIFAR10(data_root, train=False, download=False, transform=resnet_eval_tf), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print("Data ready")

In [ ]:
# cell 3 — Scratch CNN architecture
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total_seen = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_seen += x.size(0)
    return total_loss / total_seen, total_correct / total_seen

def fit(model, train_loader, val_loader, epochs, optimizer):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        with torch.no_grad():
            val_loss, val_acc = run_epoch(model, val_loader)
        for key, value in [("train_loss", train_loss), ("train_acc", train_acc), ("val_loss", val_loss), ("val_acc", val_acc)]:
            history[key].append(float(value))
        print(f"{epoch:02d}/{epochs} train_acc={train_acc:.3f} val_acc={val_acc:.3f} time={time.time()-start:.1f}s")
    return history

def evaluate(model, loader, max_misclassified=30):
    model.eval()
    confusion = torch.zeros(10, 10, dtype=torch.int64)
    total_correct, total_seen = 0, 0
    misclassified = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            probs = logits.softmax(1)
            pred = probs.argmax(1)
            total_correct += (pred == y).sum().item()
            total_seen += x.size(0)
            for true, guessed in zip(y.cpu(), pred.cpu()):
                confusion[int(true), int(guessed)] += 1
            wrong = (pred != y).nonzero(as_tuple=False).flatten()
            for idx in wrong[: max(0, max_misclassified - len(misclassified))]:
                misclassified.append({
                    "true": CLASSES[int(y[idx])],
                    "pred": CLASSES[int(pred[idx])],
                    "confidence": float(probs[idx, pred[idx]].detach().cpu()),
                    "image_path": f"miss_{len(misclassified):03d}.png",
                })
    per_class = (confusion.diag() / confusion.sum(1).clamp_min(1)).tolist()
    return {
        "test_acc": total_correct / total_seen,
        "confusion": confusion.tolist(),
        "per_class_acc": {name: float(per_class[i]) for i, name in enumerate(CLASSES)},
        "misclassifications": misclassified,
    }

In [ ]:
# cell 4 — Scratch CNN training
scratch = SimpleCNN().to(DEVICE)
scratch_optimizer = torch.optim.Adam(scratch.parameters(), lr=1e-3)
scratch_history = fit(scratch, scratch_train_loader, scratch_val_loader, epochs=20, optimizer=scratch_optimizer)

In [ ]:
# cell 5 — Scratch CNN evaluation
scratch_eval = evaluate(scratch, scratch_test_loader)
scratch_result = {
    "name": "scratch",
    "epochs": 20,
    "batch_size": 128,
    "optimizer": "Adam",
    "lr": 1e-3,
    "trainable_params": trainable_params(scratch),
    **scratch_history,
    **scratch_eval,
}
torch.save(scratch.state_dict(), "scratch-cnn.pt")
print("Scratch test_acc", scratch_result["test_acc"])

In [ ]:
# cell 6 — ResNet-18 feature extractor setup
def make_feature_extractor():
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(DEVICE)

feature_extractor = make_feature_extractor()
feature_optimizer = torch.optim.Adam(feature_extractor.fc.parameters(), lr=1e-3)
print("trainable", trainable_params(feature_extractor))

In [ ]:
# cell 7 — Feature extractor training
feature_history = fit(feature_extractor, resnet_train_loader, resnet_val_loader, epochs=10, optimizer=feature_optimizer)

In [ ]:
# cell 8 — Feature extractor evaluation
feature_eval = evaluate(feature_extractor, resnet_test_loader)
feature_result = {
    "name": "feature_extractor",
    "epochs": 10,
    "batch_size": 64,
    "optimizer": "Adam",
    "lr": 1e-3,
    "trainable_params": trainable_params(feature_extractor),
    **feature_history,
    **feature_eval,
}
print("Feature extractor test_acc", feature_result["test_acc"])

In [ ]:
# cell 9 — Fine-tune setup
fine_tune = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in fine_tune.parameters():
    p.requires_grad = False
for p in fine_tune.layer4.parameters():
    p.requires_grad = True
fine_tune.fc = nn.Linear(fine_tune.fc.in_features, 10)
fine_tune = fine_tune.to(DEVICE)
fine_optimizer = torch.optim.Adam([
    {"params": fine_tune.layer4.parameters(), "lr": 1e-4},
    {"params": fine_tune.fc.parameters(), "lr": 1e-3},
])
print("trainable", trainable_params(fine_tune))

In [ ]:
# cell 10 — Fine-tune training
fine_history = fit(fine_tune, resnet_train_loader, resnet_val_loader, epochs=10, optimizer=fine_optimizer)

In [ ]:
# cell 11 — Fine-tune evaluation
fine_eval = evaluate(fine_tune, resnet_test_loader)
fine_result = {
    "name": "fine_tune",
    "epochs": 10,
    "batch_size": 64,
    "optimizer": "Adam",
    "lr": {"layer4": 1e-4, "fc": 1e-3},
    "trainable_params": trainable_params(fine_tune),
    **fine_history,
    **fine_eval,
}
print("Fine-tune test_acc", fine_result["test_acc"])

In [ ]:
# cell 12 — Export results
results = {
    "scratch": scratch_result,
    "feature_extractor": feature_result,
    "fine_tune": fine_result,
}
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(json.dumps({
    name: {
        "test_acc": round(run["test_acc"], 4),
        "trainable_params": run["trainable_params"],
    }
    for name, run in results.items()
}, indent=2))
try:
    from google.colab import files
    files.download("results.json")
    files.download("scratch-cnn.pt")
except Exception:
    print("results.json and scratch-cnn.pt saved locally")